In [ ]:
!pip install faiss-cpu sentence-transformers


In [ ]:
import pandas as pd
import numpy as np
import os
import faiss
import pickle
from sentence_transformers import SentenceTransformer
import torch
from tqdm.notebook import tqdm

# --- 1. Kaggle Paths & Setup ---
# Kaggle working directory where we saved the Phase 1 output
# You will need to "Add Data" in Kaggle and select the notebook from Phase 1.
# Usually, it will be at: /kaggle/input/YOUR_PHASE_1_NOTEBOOK_NAME/preprocessed_data.pkl
INPUT_DATA_PATH = '/kaggle/input/notebooks/minatahmasebi/preprocessing/preprocessed_data.pkl' # UPDATE THIS
OUTPUT_PATH = '/kaggle/working/'

# Ensure GPU is used if available (T4 in Kaggle)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# --- 2. Load Preprocessed Data ---
print("Loading data from Phase 1...")
df = pd.read_pickle(INPUT_DATA_PATH)
print(f"Loaded {df.shape[0]} rows.")

# For the Knowledge Base, we only want to retrieve SARCASTIC examples (label == 1)
# The paper aims to provide sarcastic examples as context to the model
kb_df = df[df['label'] == 1].reset_index(drop=True)
print(f"Knowledge Base size (Sarcastic only): {kb_df.shape[0]}")

# --- 3. Initialize Embedding Model ---
# all-MiniLM-L6-v2 is extremely fast and provides great sentence embeddings
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
print(f"Loading {model_name}...")
model = SentenceTransformer(model_name, device=device)

# --- 4. Generate Embeddings ---
# We use the 'combined_text' (Context + Reply) created in Phase 1
print("Generating embeddings for the Knowledge Base...")
# This will take some time on ~500k rows, but T4 handles MiniLM well.
# Batch size can be increased based on T4 VRAM (16GB)
embeddings = model.encode(kb_df['combined_text'].tolist(), 
                          batch_size=512, 
                          show_progress_bar=True,
                          convert_to_numpy=True)

# --- 5. Build FAISS Index ---
print("Building FAISS Index...")
embedding_dim = embeddings.shape[1]

# IndexFlatIP uses Inner Product (Cosine Similarity if vectors are normalized)
# SentenceTransformers outputs normalized vectors for all-MiniLM
index = faiss.IndexFlatIP(embedding_dim)

# Move index to GPU for even faster processing if needed (optional for index building)
# res = faiss.StandardGpuResources()
# gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
# gpu_index.add(embeddings)
# index = faiss.index_gpu_to_cpu(gpu_index)

index.add(embeddings)
print(f"Index built with {index.ntotal} vectors.")

# --- 6. Save Artifacts for Phase 2B ---
print("Saving FAISS index and KB dataset...")
faiss.write_index(index, os.path.join(OUTPUT_PATH, 'sarcasm_kb.faiss'))
kb_df.to_pickle(os.path.join(OUTPUT_PATH, 'kb_dataset.pkl'))

# Also save the full dataset for the next step so we don't need the Phase 1 input again
df.to_pickle(os.path.join(OUTPUT_PATH, 'full_dataset.pkl'))

print("Phase 2A Complete! You can now move to Phase 2B.")
